In [1]:
import sys

print(sys.executable)
print(sys.version)

/home/ayxiao227/dft_ml_project/.venv/bin/python
3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]


In [41]:
import pubchempy as pcp
import pandas as pd
import regex as re


In [68]:
properties = ["IUPACName", "MolecularFormula", "CanonicalSMILES", "Charge", "HeavyAtomCount"]

In [67]:
molecules = ["methane", "ethanol", "acetone", "ammonia"]

In [69]:
constraints = {
    "max_molecular_weight": 150,
    "max_heavy_atoms": 10,
    "allowed_elements": {"C", "H", "O", "N"},
    "neutral_only": True
}

Formula to elements

In [70]:
def get_elements_from_smiles_regex(smiles):
    # bracket atoms: [ ... ]  |  two-letter organic subset: Br, Cl  |  single-letter (incl. aromatic)
    token_re = re.compile(r'\[([^\]]+)\]|(Br|Cl|[BCNOPSFI]|[bcnops])')

    elements = set()
    for m in token_re.finditer(smiles):
        bracket_content, plain = m.group(1), m.group(2)
        if bracket_content is not None:
            # inside brackets: optional isotope digits, then the element symbol
            mm = re.match(r'^\d*([A-Za-z][a-z]?)', bracket_content)
            if mm:
                sym = mm.group(1)
                elements.add(sym.capitalize())
        else:
            elements.add(plain.capitalize())
    return elements


In [57]:
print(get_elements_from_smiles_regex("ccccconn"))

{'N', 'O', 'C'}


In [62]:
print(get_elements_from_smiles_regex("ccccconn").issubset(constraints["allowed_elements"]))

True


In [72]:
print(properties)
print(molecules)

['IUPACName', 'MolecularFormula', 'CanonicalSMILES', 'Charge', 'HeavyAtomCount']
['methane', 'ethanol', 'acetone', 'ammonia']


In [86]:
result = pcp.get_properties(properties, "methane", "name")

In [85]:
print(result)

    MolecularFormula ConnectivitySMILES IUPACName  Charge  HeavyAtomCount
CID                                                                      
297              CH4                  C   methane       0               1


In [87]:
print(result[0]["CID"])

297


In [107]:
allowed_elements = ["C", "H"]

In [93]:
print(result)

[]


In [108]:
data = []
valid_data = []
for name in molecules:
    result = pcp.get_properties(properties, name, "name")
    if result:
        data.append(result)
        if get_elements_from_smiles_regex(result[0]["ConnectivitySMILES"]).issubset(allowed_elements) and result[0]["Charge"] == 0 and result[0]["HeavyAtomCount"] < 10:
            valid_data.append(result)
    else:
        print(f"Warning: Could not find '{name}' in PubChem.")

In [109]:
print(data)
print(valid_data)

[[{'CID': 297, 'MolecularFormula': 'CH4', 'ConnectivitySMILES': 'C', 'IUPACName': 'methane', 'Charge': 0, 'HeavyAtomCount': 1}], [{'CID': 702, 'MolecularFormula': 'C2H6O', 'ConnectivitySMILES': 'CCO', 'IUPACName': 'ethanol', 'Charge': 0, 'HeavyAtomCount': 3}], [{'CID': 180, 'MolecularFormula': 'C3H6O', 'ConnectivitySMILES': 'CC(=O)C', 'IUPACName': 'propan-2-one', 'Charge': 0, 'HeavyAtomCount': 4}], [{'CID': 222, 'MolecularFormula': 'H3N', 'ConnectivitySMILES': 'N', 'IUPACName': 'azane', 'Charge': 0, 'HeavyAtomCount': 1}]]
[[{'CID': 297, 'MolecularFormula': 'CH4', 'ConnectivitySMILES': 'C', 'IUPACName': 'methane', 'Charge': 0, 'HeavyAtomCount': 1}]]


In [104]:
flat_list = [item for sublist in data for item in sublist]

In [105]:
print(flat_list)

[{'CID': 297, 'MolecularFormula': 'CH4', 'ConnectivitySMILES': 'C', 'IUPACName': 'methane', 'Charge': 0, 'HeavyAtomCount': 1}, {'CID': 702, 'MolecularFormula': 'C2H6O', 'ConnectivitySMILES': 'CCO', 'IUPACName': 'ethanol', 'Charge': 0, 'HeavyAtomCount': 3}, {'CID': 180, 'MolecularFormula': 'C3H6O', 'ConnectivitySMILES': 'CC(=O)C', 'IUPACName': 'propan-2-one', 'Charge': 0, 'HeavyAtomCount': 4}, {'CID': 222, 'MolecularFormula': 'H3N', 'ConnectivitySMILES': 'N', 'IUPACName': 'azane', 'Charge': 0, 'HeavyAtomCount': 1}]


In [106]:
df = pd.DataFrame(flat_list)

print(df)

   CID MolecularFormula ConnectivitySMILES     IUPACName  Charge  \
0  297              CH4                  C       methane       0   
1  702            C2H6O                CCO       ethanol       0   
2  180            C3H6O            CC(=O)C  propan-2-one       0   
3  222              H3N                  N         azane       0   

   HeavyAtomCount  
0               1  
1               3  
2               4  
3               1  
